# 03 Qwen3-32B (thinking)
Reasoning on (temperature 0.6, top_p 0.95, top_k 20, 16384 tokens), pinned provider. Preflight must pass before the full run.

In [8]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display
from common import config, env, data, preflight
from common.config import RunConfig, resolve_limit, load_api_key
from common.llm_client import OpenRouterClient, fetch_endpoints
from common.pipeline import run_model_pipeline

In [9]:
# Run_Config (single source of truth for this notebook)
RUN_NAME = "qwen3_32b_thinking"
MODEL_ID = "qwen/qwen3-32b"
LIMIT = resolve_limit(None)  # None = full dev set; or set A1_LIMIT
PROVIDER_ORDER = ("DeepInfra",)  # REQUIRED, e.g. ("DeepInfra",) -- pick from the fetch_endpoints table below

def make_cfg():
    if not PROVIDER_ORDER:
        raise ValueError("PROVIDER_ORDER must be set for the reasoning run: choose a provider "
                         "from the fetch_endpoints table and set PROVIDER_ORDER = ('Name',).")
    return RunConfig(run_name=RUN_NAME, model_id=MODEL_ID, temperature=0.6, top_p=0.95,
                     top_k=20, max_tokens=16384, reasoning_enabled=True,
                     provider_order=tuple(PROVIDER_ORDER), allow_fallbacks=False, limit=LIMIT)

print("MODEL_ID", MODEL_ID, "LIMIT", LIMIT, "PROVIDER_ORDER", PROVIDER_ORDER)

MODEL_ID qwen/qwen3-32b LIMIT None PROVIDER_ORDER ('DeepInfra',)


## Provider endpoints

In [10]:
# Available providers + reported quantization; choose PROVIDER_ORDER from this table.
endpoints = fetch_endpoints(MODEL_ID)
display(pd.DataFrame(endpoints))

,provider_name,quantization,context_length,pricing
0,DeepInfra,fp8,40960,"{'prompt': '0.00000008', 'completion': '0.0000..."
1,Tenstorrent,fp8,41000,"{'prompt': '0.00000014', 'completion': '0.0000..."
2,SiliconFlow,fp8,131072,"{'prompt': '0.00000014', 'completion': '0.0000..."


## Preflight gate

In [11]:
client = OpenRouterClient(api_key=load_api_key(required=True))

In [12]:
# Preflight on the first min(preflight_n, N) examples; the full run is blocked unless it passes.
cfg = make_cfg()
examples = data.load_examples(LIMIT)
with env.track_run(RUN_NAME + ":preflight", LIMIT, extra={"provider_order": list(cfg.provider_order)}):
    try:
        pf = preflight.run_preflight(cfg, examples, client)
    except preflight.PreflightFailedError as e:
        rep = e.report
        failing = [c for c in rep.checks if not c["passed"]]
        display(pd.DataFrame(failing))
        raise SystemExit(f"Preflight FAILED -- full run not started. Report: {e.report_path}. "
                         "Fix the provider pin or config and re-run this cell.")
display(pd.DataFrame(pf.checks))
print("Preflight passed; providers served:", pf.providers_served)

,name,passed,failing_ids,detail
0,api_ok,True,[],status == 'ok'
1,reasoning_nonempty,True,[],non-empty raw reasoning text
2,reasoning_raw_not_summary,True,[],"raw reasoning, not summary/encrypted-only"
3,reasoning_length_consistent,True,[],len(reasoning)/reasoning_tokens >= min_chars_p...
4,reasoning_tokens_positive,True,[],reasoning_tokens > 0
5,sql_extractable,True,[],"Extraction_Status in {ok, fallback}"
6,provider_matches_pin,True,[],served provider within cfg.provider_order


Preflight passed; providers served: ['DeepInfra']


## Full run, extract, predict, evaluate, Official_CLI check

In [13]:
cfg = make_cfg()
with env.track_run(RUN_NAME, LIMIT, extra={"fingerprint": cfg.fingerprint()}):
    res = run_model_pipeline(cfg, client, endpoints)
display(res.eval.table)
print("missing records:", res.n_missing, "| CLI check passed:", not res.mismatches)

qwen3_32b_thinking[full]:   0%|          | 0/1034 [00:00<?, ?it/s]

[pipeline] run: RunStats(n_total=1034, n_skipped=0, n_ok=1034, n_error=0, providers=Counter({'DeepInfra': 1034}), bad_lines=[], stopped_early=False)
[pipeline] wrapper totals match the Official_CLI.


,count,ex,em
easy,248,0.899194,0.616935
medium,446,0.827354,0.345291
hard,174,0.706897,0.252874
extra,166,0.566265,0.054217
all,1034,0.782398,0.348162


missing records: 0 | CLI check passed: True


## Provider report

In [14]:
# Served providers vs. the pin, with reported quantization (Req 12.5, 12.6, 22.5)
display(res.providers)
if (res.providers["flag"] == "outside pin").any():
    print("WARNING: some calls were served by providers outside the pin.")

,provider,calls,share,quantization,in_pin,flag
0,DeepInfra,1034,1.0,fp8,True,
